# Notebook 03: Model Selection Framework

This notebook provides a structured framework for selecting the right LLM for your use case, including cost estimation, latency benchmarking, and building comparison tables.

**Learning Objectives:**
- Apply a decision framework for model selection
- Calculate and compare costs across models
- Benchmark latency for different models
- Build a model comparison table for a specific use case

## 1. Model Selection Decision Framework

In [ ]:
def evaluate_model(model_name, requirements):
    """
    Score a model against requirements.
    Each requirement is scored 1-5. Model is scored similarly.
    Total score = sum of (requirement_weight * match_score)
    """
    scores = {}
    
    # Model capability ratings (simplified)
    model_profiles = {
        "gpt-4o": {
            "quality": 5, "speed": 4, "cost": 3, "privacy": 1,
            "context": 5, "multilingual": 5, "code": 5
        },
        "gpt-4o-mini": {
            "quality": 4, "speed": 5, "cost": 5, "privacy": 1,
            "context": 4, "multilingual": 4, "code": 4
        },
        "claude-sonnet-4": {
            "quality": 5, "speed": 4, "cost": 3, "privacy": 2,
            "context": 5, "multilingual": 4, "code": 5
        },
        "claude-haiku-3.5": {
            "quality": 4, "speed": 5, "cost": 4, "privacy": 2,
            "context": 4, "multilingual": 3, "code": 4
        },
        "gpt-4o": {
            "quality": 5, "speed": 4, "cost": 3, "privacy": 2,
            "context": 5, "multilingual": 5, "code": 4
        },
        "gpt-4o-mini": {
            "quality": 4, "speed": 5, "cost": 5, "privacy": 2,
            "context": 5, "multilingual": 4, "code": 4
        },
        "llama-3.1-70b": {
            "quality": 4, "speed": 3, "cost": 4, "privacy": 5,
            "context": 4, "multilingual": 4, "code": 4
        },
        "llama-3.1-8b": {
            "quality": 3, "speed": 5, "cost": 5, "privacy": 5,
            "context": 4, "multilingual": 3, "code": 3
        },
        "deepseek-v3": {
            "quality": 4, "speed": 4, "cost": 5, "privacy": 3,
            "context": 4, "multilingual": 4, "code": 5
        }
    }
    
    if model_name not in model_profiles:
        return None
    
    profile = model_profiles[model_name]
    total_score = 0
    total_weight = 0
    
    for criterion, (score, weight) in requirements.items():
        if criterion in profile:
            # How well does the model match this requirement?
            match = min(profile[criterion], score) / max(profile[criterion], score)
            total_score += match * weight
            total_weight += weight
    
    return total_score / total_weight if total_weight > 0 else 0

# Example: evaluating models for a customer support chatbot
print("=== Customer Support Chatbot Evaluation ===")
print("Requirements: good quality (4), fast (5), cheap (5), moderate privacy (3)\n")

requirements = {
    "quality": (4, 0.3),     # (required_score, weight)
    "speed": (5, 0.25),
    "cost": (5, 0.25),
    "privacy": (3, 0.2)
}

results = []
for model in ["gpt-4o", "gpt-4o-mini", "claude-sonnet-4", "claude-haiku-3.5",
              "gpt-4o-mini", "llama-3.1-70b", "llama-3.1-8b", "deepseek-v3"]:
    score = evaluate_model(model, requirements)
    results.append({"Model": model, "Score": round(score, 3) if score else 0})

import pandas as pd
results_df = pd.DataFrame(results).sort_values("Score", ascending=False)
display(results_df.style.bar(subset=["Score"], color="#5fba7d"))

## 2. Cost Estimation Calculator

In [ ]:
# Model pricing data ($/1M tokens)
MODEL_PRICING = {
    "gpt-4o": {"input": 2.50, "output": 10.00},
    "gpt-4o-mini": {"input": 0.15, "output": 0.60},
    "claude-opus-4": {"input": 15.00, "output": 75.00},
    "claude-sonnet-4": {"input": 3.00, "output": 15.00},
    "claude-haiku-3.5": {"input": 0.80, "output": 4.00},
    "gpt-4o": {"input": 1.25, "output": 10.00},
    "gpt-4o-mini": {"input": 0.15, "output": 0.60},
    "deepseek-v3": {"input": 0.27, "output": 1.10},
}

def calculate_monthly_cost(
    model: str,
    daily_requests: int,
    avg_input_tokens: int,
    avg_output_tokens: int,
    days: int = 30
) -> dict:
    """Calculate monthly cost for a given model and usage pattern."""
    pricing = MODEL_PRICING.get(model)
    if not pricing:
        return {"error": f"Unknown model: {model}"}
    
    total_input = daily_requests * avg_input_tokens * days
    total_output = daily_requests * avg_output_tokens * days
    
    input_cost = (total_input / 1_000_000) * pricing["input"]
    output_cost = (total_output / 1_000_000) * pricing["output"]
    
    return {
        "Model": model,
        "Monthly Input Tokens": f"{total_input:,.0f}",
        "Monthly Output Tokens": f"{total_output:,.0f}",
        "Input Cost": f"${input_cost:.2f}",
        "Output Cost": f"${output_cost:.2f}",
        "Total Monthly Cost": f"${input_cost + output_cost:.2f}",
        "Cost per Request": f"${(input_cost + output_cost) / (daily_requests * days):.6f}"
    }

# Example: SaaS chatbot with 10,000 requests/day
print("=== SaaS Chatbot Cost Comparison (10K requests/day) ===")
print("Each request: 200 input tokens, 300 output tokens\n")

costs = []
for model in MODEL_PRICING:
    result = calculate_monthly_cost(model, 10_000, 200, 300)
    costs.append(result)

display(pd.DataFrame(costs).set_index("Model"))

### Scaling Cost Analysis

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Cost at different scales
scales = [1_000, 5_000, 10_000, 50_000, 100_000]  # daily requests
models_to_plot = ["gpt-4o-mini", "claude-haiku-3.5", "gpt-4o-mini", "deepseek-v3"]

fig, ax = plt.subplots(figsize=(10, 6))

for model in models_to_plot:
    monthly_costs = []
    for scale in scales:
        pricing = MODEL_PRICING[model]
        total_input = scale * 200 * 30
        total_output = scale * 300 * 30
        cost = (total_input / 1_000_000) * pricing["input"] + (total_output / 1_000_000) * pricing["output"]
        monthly_costs.append(cost)
    ax.plot(scales, monthly_costs, marker='o', label=model)

ax.set_xlabel("Daily Requests")
ax.set_ylabel("Monthly Cost ($)")
ax.set_title("Monthly Cost at Different Scales (200 input + 300 output tokens/request)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Latency Benchmarking

Measure time-to-first-token (TTFT) and tokens-per-second for different models.

In [ ]:
import time
import os
from dotenv import load_dotenv

load_dotenv()

OPENAI_KEY = os.getenv("OPENAI_API_KEY")

def benchmark_model(model_name, prompt, n_runs=3):
    """
    Benchmark a model's latency.
    Returns average TTFT, total time, and tokens per second.
    """
    if not OPENAI_KEY:
        return {"error": "No API key available"}
    
    from openai import OpenAI
    client = OpenAI(api_key=OPENAI_KEY)
    
    ttfts = []
    total_times = []
    token_counts = []
    
    for _ in range(n_runs):
        start = time.time()
        first_token_time = None
        full_response = ""
        
        stream = client.chat.completions.create(
            model=model_name,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=200,
            stream=True
        )
        
        for chunk in stream:
            if chunk.choices[0].delta.content:
                if first_token_time is None:
                    first_token_time = time.time()
                full_response += chunk.choices[0].delta.content
        
        end = time.time()
        
        ttfts.append(first_token_time - start if first_token_time else 0)
        total_times.append(end - start)
        # Rough token count
        token_counts.append(len(full_response.split()) * 1.3)  # approximate
    
    avg_ttft = sum(ttfts) / len(ttfts)
    avg_total = sum(total_times) / len(total_times)
    avg_tps = sum(token_counts) / avg_total if avg_total > 0 else 0
    
    return {
        "Model": model_name,
        "Avg TTFT (s)": round(avg_ttft, 3),
        "Avg Total Time (s)": round(avg_total, 3),
        "Est. Tokens/sec": round(avg_tps, 1)
    }

# Benchmark prompt
bench_prompt = "Write a detailed explanation of how a neural network learns, covering forward pass, loss computation, and backpropagation."

print("Benchmarking models...")
print("This may take a minute if you have API keys configured.\n")

if OPENAI_KEY:
    models_to_bench = ["gpt-4o-mini", "gpt-4o"]
    benchmark_results = []
    for model in models_to_bench:
        print(f"Benchmarking {model}...")
        result = benchmark_model(model, bench_prompt, n_runs=2)
        benchmark_results.append(result)
    
    display(pd.DataFrame(benchmark_results).set_index("Model"))
else:
    print("No API key. Example results (illustrative):")
    example_results = [
        {"Model": "gpt-4o-mini", "Avg TTFT (s)": 0.35, "Avg Total Time (s)": 2.1, "Est. Tokens/sec": 95},
        {"Model": "gpt-4o", "Avg TTFT (s)": 0.55, "Avg Total Time (s)": 4.8, "Est. Tokens/sec": 55},
    ]
    display(pd.DataFrame(example_results).set_index("Model"))

## 4. Building a Complete Model Comparison Table

In [ ]:
# Comprehensive model comparison
comparison_data = pd.DataFrame({
    "Model": [
        "GPT-4o", "GPT-4o-mini",
        "Claude Opus 4", "Claude Sonnet 4", "Claude Haiku 3.5",
        "GPT-4o", "GPT-4o Mini",
        "Llama 3.1 70B", "Llama 3.1 8B",
        "DeepSeek-V3"
    ],
    "Type": ["API"] * 7 + ["Self-Host"] * 3,
    "Context (tokens)": ["128K", "128K", "200K", "200K", "200K", "1M+", "1M+", "128K", "128K", "128K"],
    "Input $/1M": [2.50, 0.15, 15.00, 3.00, 0.80, 1.25, 0.15, 0, 0, 0],
    "Output $/1M": [10.00, 0.60, 75.00, 15.00, 4.00, 10.00, 0.60, 0, 0, 0],
    "Quality (1-5)": [5, 4, 5, 5, 4, 5, 4, 4, 3, 4],
    "Speed (1-5)": [4, 5, 3, 4, 5, 4, 5, 3, 5, 4],
    "Code (1-5)": [5, 4, 5, 5, 4, 4, 4, 4, 3, 5],
    "Privacy (1-5)": [1, 1, 2, 2, 2, 2, 2, 5, 5, 3],
    "Best For": [
        "Complex tasks, reasoning",
        "High-volume, cost-sensitive",
        "Long documents, analysis",
        "Balanced quality/cost",
        "Quick tasks, budget",
        "Huge context, multimodal",
        "Fast, cheap, huge context",
        "Custom fine-tuning",
        "Edge, prototyping",
        "Code, cost-effective"
    ]
})

display(comparison_data.style.set_properties(**{'text-align': 'left'}).set_table_styles([
    {'selector': 'th', 'props': [('text-align', 'left')]}
]))

### Use-Case Specific Recommendations

In [ ]:
use_case_recommendations = pd.DataFrame({
    "Use Case": [
        "Customer Support Chatbot",
        "Document Summarization",
        "Code Generation",
        "Content at Scale",
        "Privacy-Sensitive App",
        "Complex Analysis",
        "Edge Deployment"
    ],
    "Primary Pick": [
        "GPT-4o-mini",
        "Claude Sonnet 4",
        "GPT-4o",
        "DeepSeek-V3",
        "Llama 3.1 70B",
        "Claude Opus 4",
        "Llama 3.1 8B"
    ],
    "Alternative": [
        "Gemini Flash",
        "Gemini Pro",
        "Claude Sonnet 4",
        "GPT-4o-mini",
        "Qwen 2.5 72B",
        "GPT-4o",
        "Phi-3.5 Mini"
    ],
    "Why": [
        "Best cost/quality ratio for high volume",
        "200K context handles long docs",
        "Strong reasoning and code training",
        "Lowest API cost, good quality",
        "Full control, no data sharing",
        "Best reasoning, worth the premium",
        "Small enough for CPU/edge inference"
    ]
})

display(use_case_recommendations.style.set_properties(**{'text-align': 'left'}))

## 5. Decision Worksheet

Use this template to document your model selection process for a real project.

In [ ]:
# Fill in for your specific use case
decision_worksheet = {
    "Project": "[Your project name]",
    "Use Case": "[e.g., Customer support chatbot]",
    "Requirements": {
        "Quality threshold": "[1-5]",
        "Monthly token volume": "[estimate]",
        "Budget": "[$/month]",
        "Latency requirement": "[e.g., < 2s TTFT]",
        "Privacy level": "[none / moderate / strict]",
        "Languages": "[English / multilingual]",
        "Code generation": "[yes / no]"
    },
    "Top 3 Candidates": [
        {"Model": "", "Score": "", "Reason": ""},
        {"Model": "", "Score": "", "Reason": ""},
        {"Model": "", "Score": "", "Reason": ""}
    ],
    "Cost Estimate": "",
    "Final Decision": "",
    "Rationale": ""
}

for key, value in decision_worksheet.items():
    if isinstance(value, dict):
        print(f"{key}:")
        for k, v in value.items():
            print(f"  {k}: {v}")
    elif isinstance(value, list):
        print(f"{key}:")
        for item in value:
            print(f"  {item}")
    else:
        print(f"{key}: {value}")

## Key Takeaways

1. **No one-size-fits-all** — model selection depends on your specific requirements
2. **Score systematically** — use weighted criteria to avoid bias toward familiar models
3. **Cost scales non-linearly** — cheap models at high volume can outperform expensive models at low volume
4. **Benchmark on YOUR data** — public benchmarks don't guarantee performance on your tasks
5. **Document your decision** — future you (and your team) will thank you